# LLM Architectures Explained: Attention Mechanism (Part 5)

Code from the article [LLM Architectures Explained: Attention Mechanism (Part 5)](https://medium.com/@vipra_singh/llm-architectures-explained-attention-part-5-495bbe3d278e), in the order it appears there. Every code cell below was run from top to bottom; the outputs are from that run. Cells marked **Added for this repository** are not in the article: they download data or set things up so the notebook runs on a fresh machine.

## 1.2 Motivation

```
English => French
red => rouge
dress => robe
“red dress” => “robe rouge”

Notice how red is before dress in English but rouge is after robe.
```

## 2.2 How Does Attention Work? (worked example)

```
decoder_hidden = [10, 5, 10]

encoder_hidden  score
---------------------
     [0, 1, 1]     15 (= 10×0 + 5×1 + 10×1, the dot product)
     [5, 0, 1]     60
     [1, 1, 0]     15
     [0, 5, 1]     35
```

```
encoder_hidden  score  score^
-----------------------------
     [0, 1, 1]     15       0
     [5, 0, 1]     60       1
     [1, 1, 0]     15       0
     [0, 5, 1]     35       0
```

```
encoder  score  score^  alignment
---------------------------------
[0, 1, 1]   15      0   [0, 0, 0]
[5, 0, 1]   60      1   [5, 0, 1]
[1, 1, 0]   15      0   [0, 0, 0]
[0, 5, 1]   35      0   [0, 0, 0]
```

```
encoder  score  score^  alignment
---------------------------------
[0, 1, 1]   15     0  [0, 0, 0]
[5, 0, 1]   60     1  [5, 0, 1]
[1, 1, 0]   15     0  [0, 0, 0]
[0, 5, 1]   35     0  [0, 0, 0]
context = [0+5+0+0, 0+0+0+0, 0+1+0+0] = [5, 0, 1]
```

**Added for this repository:** computes the worked example above with NumPy, so you can change the numbers and see the result.

In [1]:
import numpy as np
# The worked example above, computed: dot-product scores, a hard (one-hot) softmax as in the article, alignment and context vectors
decoder_hidden = np.array([10, 5, 10])
encoder_hidden = np.array([[0, 1, 1], [5, 0, 1], [1, 1, 0], [0, 5, 1]])
scores = encoder_hidden @ decoder_hidden
print("scores:", scores)
score_hat = (scores == scores.max()).astype(int)      # the article rounds the softmax to 0/1 to keep the numbers simple
print("softmaxed scores (rounded):", score_hat)
print("real softmax:", np.round(np.exp(scores - scores.max()) / np.exp(scores - scores.max()).sum(), 4))
alignment = encoder_hidden * score_hat[:, None]
print("alignment vectors:\n", alignment)
print("context vector:", alignment.sum(axis=0))

scores: [15 60 15 35]
softmaxed scores (rounded): [0 1 0 0]
real softmax: [0. 1. 0. 0.]
alignment vectors:
 [[0 0 0]
 [5 0 1]
 [0 0 0]
 [0 0 0]]
context vector: [5 0 1]


## 4.5.1 Masked Multi-Head Attention

In [2]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
seq_len, d_model, d_k = 4, 8, 8
X = torch.randn(seq_len, d_model)            # 4 token embeddings

# learnable weight matrices in a real model; random here
W_q = torch.randn(d_model, d_k) / math.sqrt(d_model)
W_k = torch.randn(d_model, d_k) / math.sqrt(d_model)
W_v = torch.randn(d_model, d_k) / math.sqrt(d_model)

Q, K, V = X @ W_q, X @ W_k, X @ W_v          # Step 1: queries, keys, values
scores = Q @ K.T / math.sqrt(d_k)            # Steps 2-3: scaled dot products

# Masked (causal) attention: token i may only look at tokens 0..i
mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(mask, float("-inf"))

weights = torch.softmax(scores, dim=-1)      # Step 4: each row sums to 1
output = weights @ V                         # Step 5: weighted sum of values

print(weights.round(decimals=2))
# PyTorch's built-in version gives the same result
print(torch.allclose(output, F.scaled_dot_product_attention(Q, K, V, is_causal=True), atol=1e-5))

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.6000, 0.0000, 0.0000],
        [0.3800, 0.5900, 0.0400, 0.0000],
        [0.0900, 0.1300, 0.6800, 0.1000]])
True
